In [ ]:
import gc
import os
import sys
import random
import dotenv
import pandas as pd
import torch
from diffusers import AutoPipelineForText2Image
import matplotlib.pyplot as plt
import shutil

sys.path.append('..')
dotenv.load_dotenv()
os.environ['WANDB_DISABLED'] = "true"
assert len(os.getenv('HF_TOKEN'))>0

from vision_unlearning.unlearner import UnlearnerLoraDirect, unlearn_lora
from vision_unlearning.utils.logger import get_logger, setup_loggers
from vision_unlearning.datasets import UnlearnDatasetImagenette, UnlearnDatasetSplitMode
from vision_unlearning.utils.gradient_weighting import GradientWeightingMethodMunba
from vision_unlearning.utils.prompts import prompts

In [ ]:
num_train_epochs = 15
model_base_name = "CompVis/stable-diffusion-v1-4"
model_lora_path = f"assets/models/munba_church_{num_train_epochs:03d}"
hub_model_id = "LeonardoBenitez/demo-vision-unlearning-munba"
dataset_base_path = 'assets/datasets/imagenette_splits'

c = "n03028079"  # Church class
max_eval_prompts = 100

validation_prompt = f'An image of a church'
example_prompts_forget = [
    f'An image of a church',
    f'Photograph of a church; high definition',
]
example_prompts_retain = [
    f'An image of a house',
    f'Photograph of a old stone building; high definition',
]

### No need to change anything from now on... ###
logger = get_logger('main')
setup_loggers(modules_info=['vision_unlearning.'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'  # To stop warnings "The current process just got forked"

c_human = UnlearnDatasetImagenette.class_mapping[c]
dataset_forget_name = f"{dataset_base_path}/{c}/train_forget"
dataset_retain_name = f"{dataset_base_path}/{c}/train_retain"

# Assemble the evaluation prompts
final_eval_prompts_forget = prompts[f'imagenette_{c_human}']['forget'][:max_eval_prompts]
final_eval_prompts_retain = []
for category in prompts:
    if category.startswith('imagenette') and not category.endswith(c_human):
        final_eval_prompts_retain += prompts[category]['forget']
random.shuffle(final_eval_prompts_retain)
final_eval_prompts_retain = final_eval_prompts_retain[:len(final_eval_prompts_forget)]

# Original model
As you can see, it can generate the undesired concept

In [ ]:
pipeline = AutoPipelineForText2Image.from_pretrained(model_base_name, torch_dtype=torch.float16, safety_checker=None).to(device)

In [ ]:
for prompt in example_prompts_forget + example_prompts_retain:
    image = pipeline(prompt).images[0]
    plt.imshow(image)
    plt.title(prompt)
    plt.show()

In [ ]:
del pipeline
gc.collect()
torch.cuda.empty_cache()

# Prepare dataset for unlearning
images of the concept to be forgotten + some random images of concepts to be retained

In [ ]:
#if os.path.exists(dataset_base_path):
#    shutil.rmtree(dataset_base_path)
if not os.path.exists(f"{dataset_base_path}/{c}"):
    dataset = UnlearnDatasetImagenette(
        split_mode=UnlearnDatasetSplitMode.Class,
        split_kwargs={"forget": [c]},
        download_path=dataset_base_path,
    )
    dataset.save(f"{dataset_base_path}/{c}", format='jpg')
else:
    print('Splits already exist, not downloading again')

# Unlearning
Performing the actual model update

In [ ]:
!nvidia-smi

In [ ]:
#torch.cuda.memory._record_memory_history()

In [ ]:
free_memory, total_memory = torch.cuda.mem_get_info()  # in GB
logger.info(f"Free Memory: {free_memory / 1e9:.2f} GB")
logger.info(f"Total Memory: {total_memory / 1e9:.2f} GB")

hyperparameters = {
    "dataloader_num_workers": 2,
    "resolution": 512,
    "num_validation_images": 1,

    "mixed_precision": "no",
    "learning_rate": 15e-4,
    "max_grad_norm": 1.0,
    "lr_scheduler_type": "lr_scheduler",
    "lr_warmup_steps": 0,
    "num_train_epochs": num_train_epochs,
    "validation_epochs": 1,
    "checkpointing_steps": 10000,
    "lr_scheduler_type": "constant",
    "logging_steps": 20,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "random_flip": True,

    "lora_r": 4,
    "target_modules": ["to_k", "to_q", "to_v", "to_out.0"],
    "lora_alpha": 4,
    "lora_dropout": 0.1,

    "seed": 42,
}

if free_memory > 20e9:
    hyperparameters.update({
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 1,
    })
elif free_memory > 14e9:
    hyperparameters.update({
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 2,
    })
else:
    logger.error('Too little GPU, diverting power from life support...')
    hyperparameters.update({
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 4,
    })

logger.info(hyperparameters)

In [ ]:
unlearner = UnlearnerLoraDirect(
    model_name_or_path=model_base_name,
    dataset_forget_name=dataset_forget_name,
    dataset_retain_name=dataset_retain_name,
    output_dir=model_lora_path,
    validation_prompt=validation_prompt,
    gradient_weighting_method = GradientWeightingMethodMunba(),
    final_eval_prompts_forget = example_prompts_forget + final_eval_prompts_forget,
    final_eval_prompts_retain = final_eval_prompts_retain,
    hub_model_id = hub_model_id,
    **hyperparameters,
)

In [ ]:
eval_results = unlearner.train()
df = pd.DataFrame([{'Name': r.metric_name, 'Value': r.metric_value} for r in eval_results])
df

In [ ]:
del unlearner
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#torch.cuda.memory._dump_snapshot('assets/memory_post_del.pickle')

In [ ]:
#print(torch.cuda.memory_summary(device=0))


In [ ]:
# Show validation prompts
for epoch in range(0, num_train_epochs, 1):
    plt.imshow(plt.imread(os.path.join(model_lora_path, 'images', f'val_prompt_{epoch:02}_01.png')))
    plt.title(f'Epoch {epoch}')
    plt.show()

# Check unlearned model
as you can see, it does NOT generate the undesired concept anymore

In [ ]:
_, _, pipeline_unlearned = unlearn_lora(
    model_original_id=model_base_name,
    model_lora_id=model_lora_path,
    device=device,
    requires_inversion=True,
    return_original=False,
    return_learned=False,
)

In [ ]:
for prompt in example_prompts_forget + example_prompts_retain:
    image = pipeline_unlearned(prompt).images[0]
    plt.imshow(image)
    plt.title(prompt)
    plt.show()